APPENDIX A - INTRODUCTION TO PYTORCH

A.2 - A logistic regression forward pass

In [2]:
import torch
import torch.nn.functional as F

y = torch.tensor([1.0])
x = torch.tensor([1.1])

x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])

b = torch.tensor([0.0])
z = x1 * w1 + b

# activation & output
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

/Users/ilyanadulina/Documents/IleGPT/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


A.3 - Computing gradients via autograd

In [3]:
import torch
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [4]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


A.4 - A multilayer perceptron with two hidden layers

In [5]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # output layer
            torch.nn.Linear(20, num_outputs),
        )
    def forward(self, x):
        logits = self.layers(x)
        return logits

In [6]:
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [7]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [8]:
print(model.layers[0].weight)
print(model.layers[0].weight.shape)
print(model.layers[0].bias)

Parameter containing:
tensor([[-0.0833,  0.0240, -0.1384,  ...,  0.0299, -0.0904,  0.0982],
        [-0.1389, -0.0197, -0.1241,  ..., -0.1109, -0.0554,  0.0624],
        [ 0.1233, -0.1200, -0.1295,  ..., -0.0983, -0.0994,  0.0534],
        ...,
        [-0.0089,  0.1345, -0.0582,  ...,  0.0105,  0.1315,  0.1320],
        [-0.0390,  0.0919,  0.0199,  ...,  0.1231, -0.1057,  0.0697],
        [-0.0537, -0.0765, -0.0085,  ...,  0.0950, -0.0403, -0.1296]],
       requires_grad=True)
torch.Size([30, 50])
Parameter containing:
tensor([-0.0954, -0.0057,  0.0028,  0.0394,  0.0934,  0.1407,  0.0929, -0.0551,
         0.0491,  0.1404,  0.0595, -0.0919,  0.0968, -0.1352, -0.0763, -0.0602,
        -0.1283, -0.1127,  0.0024,  0.0026,  0.0813,  0.0127, -0.0640,  0.0212,
         0.0433,  0.0753, -0.0168, -0.0362, -0.0796,  0.1382],
       requires_grad=True)


In [9]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [ ]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)
# grad_fn=<AddmmBackward0> - last-used function to compute a variable in the computational graph

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [12]:
# best practice - inference only

with torch.no_grad():
    out = model(X)
    print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [13]:
# softmax

with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
    print(out)

tensor([[0.3113, 0.3934, 0.2952]])


A.5 - Creating a small toy dataset

In [14]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
    ])
y_test = torch.tensor([0, 1])

A.6 - Defining a custom Dataset class

In [16]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [17]:
print(len(train_ds))

5


A.7 - Instantiating data loaders

In [24]:
from torch.utils.data import DataLoader
torch.manual_seed(123)
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
)
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [25]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


A.8 - A training loader that drops the last batch

In [26]:
from torch.utils.data import DataLoader
torch.manual_seed(123)
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

In [29]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.9000,  2.9000],
        [ 2.3000, -1.1000]]) tensor([0, 1])
Batch 2: tensor([[ 2.7000, -1.5000],
        [-0.5000,  2.6000]]) tensor([1, 0])


A.9 - Neural network training in pytorch

In [30]:
import torch.nn.functional as F
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}"
        )
    model.eval()

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00


In [31]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
    print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [32]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [33]:
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [34]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [35]:
predictions == y_train

tensor([True, True, True, True, True])

In [36]:
torch.sum(predictions == y_train)

tensor(5)

A.10 - A function to compute the prediction accuracy

In [37]:
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0
    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)
    return (correct / total_examples).item()

In [38]:
print(compute_accuracy(model, train_loader))

1.0


In [39]:
print(compute_accuracy(model, test_loader))

1.0


In [40]:
torch.save(model.state_dict(), "model.pth")

In [41]:
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>

In [42]:
print(model.state_dict())

OrderedDict({'layers.0.weight': tensor([[-0.3091,  0.1047],
        [-0.3552,  0.2827],
        [-0.6039,  0.5246],
        [-0.5140, -0.5622],
        [-0.4623,  0.3811],
        [-0.2791,  0.3349],
        [-0.6001, -0.4290],
        [-0.2596, -0.1390],
        [-0.5323,  0.4353],
        [-0.1786,  0.2737],
        [ 0.5025,  0.1076],
        [ 0.1365,  0.7400],
        [-0.3292,  0.2645],
        [-0.3306,  0.5677],
        [ 0.6177, -0.6788],
        [ 0.5121,  0.3864],
        [ 0.2444,  0.3179],
        [ 0.6392, -0.1203],
        [-0.5988, -0.2008],
        [-0.5104,  0.0951],
        [-0.0307, -0.4296],
        [-0.0750,  0.7119],
        [ 0.0188, -0.0559],
        [-0.1308,  0.1919],
        [ 0.6124,  0.4841],
        [ 0.2634,  0.1395],
        [ 0.1408,  0.3752],
        [-0.0796, -0.6639],
        [-0.4392,  0.6110],
        [ 0.0745, -0.5911]]), 'layers.0.bias': tensor([ 0.1411,  0.5901, -0.6659, -0.4761, -0.2595,  0.0540, -0.1649, -0.0777,
        -0.6994,  0.3481,  0.

In [43]:
print(torch.cuda.is_available())

False


In [44]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [45]:
tensor_1 = tensor_1.to("mps")
tensor_2 = tensor_2.to("mps")
print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='mps:0')


A.11 - A training loop on a GPU

In [46]:
torch.manual_seed(123)
model = NeuralNetwork(2,2)
device = torch.device("mps")
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(),lr=0.5)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features,labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
        model.eval()

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00
